# Listening test preparation

Turns the baseline's recommendations into a blind listening test that several
people can rate.

The baseline already has a manual evaluation: 10 seed tracks, 5 recommendations
each. Rating those tells us what people think of the recommendations, but not
whether they think more of them than of nothing — every one of the 50 pairs
comes from the model, so there is no floor to read the scores against. Three
things are added here:

- **random pairs**, one per seed, so a score has a control to be compared with.
  This is the same argument as the random baseline for genre agreement.
- **deep pairs**, drawn from further down the ranking, so we can see where a
  listener stops hearing two tracks as related. That point is the far end of
  the adventure slider, which is currently set by hand.
- **blind labelling**: the rater sees a pair number and two clips. No artist,
  no title, no genre, no cosine score. Seeing "Folk → Folk" nudges a rating
  upward without the rater noticing.

### What this writes

```
data/listening_test/
    clips/P01_A.mp3, P01_B.mp3, ...   the audio, blind names
    manifest.csv                      what the rating form needs
    assignments.csv                   which rater gets which pairs
    pairs_key.csv                     ANSWER KEY — never give this to a rater
```

`data/` is gitignored, so none of it is committed. Re-run the notebook to
rebuild it; the random draws are seeded, so it rebuilds identically.

Nothing else in the repo is touched, and this notebook imports nothing from
`modeling/` — it is self-contained so it runs in either of our checkouts.

In [1]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

# our two checkouts keep the data in different places, so look rather than assume
def find_dir(*candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(f"none of these exist: {candidates}")

AUDIO_DIR = find_dir("../data/fma_small", "../data/raw/fma_small", "data/fma_small")
META_DIR = find_dir("../data/fma_metadata", "../data/raw/fma_metadata", "data/fma_metadata")
OUT_DIR = AUDIO_DIR.parent / "listening_test"
CLIPS_DIR = OUT_DIR / "clips"

SEED = 42  # every random draw below uses this, so the test rebuilds identically
rng = np.random.default_rng(SEED)

print(f"audio  : {AUDIO_DIR}")
print(f"meta   : {META_DIR}")
print(f"output : {OUT_DIR}")


def audio_path(track_id: int) -> Path:
    """FMA stores 000140.mp3 under a folder named by its first three digits."""
    name = f"{track_id:06d}"
    return AUDIO_DIR / name[:3] / f"{name}.mp3"

audio  : ../data/fma_small
meta   : ../data/fma_metadata
output : ../data/listening_test


## The baseline's recommendations

The 50 pairs from the manual evaluation, as track ids. Kept here as literal
data rather than regenerated, so this notebook rates *exactly* the pairs the
baseline produced, and does not silently drift if the model changes.

In [2]:
MODEL_PAIRS = {
    140:    [44805, 6803, 77564, 110451, 52953],
    54033:  [145556, 63456, 14571, 114070, 134388],
    59876:  [35544, 141617, 36959, 64009, 44950],
    42234:  [136137, 36273, 36277, 44821, 44820],
    68893:  [133028, 72071, 131796, 100552, 51301],
    43883:  [66482, 112668, 38907, 38905, 48368],
    123978: [1069, 115850, 92283, 997, 114279],
    149701: [107048, 130986, 29720, 12690, 126773],
    63471:  [145199, 86417, 54942, 73521, 3833],
    17608:  [43028, 116706, 68840, 39663, 68841],
}

SEEDS = list(MODEL_PAIRS)
print(f"{len(SEEDS)} seeds, {sum(len(v) for v in MODEL_PAIRS.values())} model pairs")

# 35544 and 36959 are the same Garmisch track in two versions, so one of seed
# 59876's five slots is spent twice. Left in deliberately: this is the list the
# baseline produced, and the duplicate is a finding, not something to hide.
missing = [t for t in {t for s, r in MODEL_PAIRS.items() for t in [s, *r]}
           if not audio_path(t).exists()]
print("missing audio:", missing or "none")

10 seeds, 50 model pairs
missing audio: none


In [3]:
tracks = pd.read_csv(META_DIR / "tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[("set", "subset")] == "small"]
print(f"{len(small)} tracks in the small subset")


def label(track_id: int) -> dict:
    """Artist, title and genre — for the answer key only, never for the rater."""
    row = small.loc[track_id]
    return {
        "artist": row[("artist", "name")],
        "title": row[("track", "title")],
        "genre": row[("track", "genre_top")],
    }

8000 tracks in the small subset


## Random pairs — the control

One random partner per seed, drawn from the rest of the catalogue.

Using the *same* seeds as the model pairs is deliberate: it makes the
comparison paired. Some seed tracks are simply easier to match than others, and
pairing removes that difference from the result instead of leaving it as noise.

In [4]:
recommended = {t for recs in MODEL_PAIRS.values() for t in recs}
pool = [t for t in small.index if t not in recommended and t not in SEEDS]

random_pairs = {s: int(rng.choice(pool)) for s in SEEDS}
for s, other in random_pairs.items():
    print(f"{label(s)['artist'][:28]:30} <-> {label(other)['artist'][:28]:30} "
          f"[{label(other)['genre']}]")

Alec K. Redfearn & the Eyeso   <-> Jonathan Coulton               [Folk]
alright lover                  <-> Six Umbrellas                  [Electronic]
Boy Friend                     <-> Mc Norman                      [International]
Zlatne Uste Balkan Brass Ban   <-> Misay Day                      [Folk]
Broke For Free                 <-> Square People                  [Rock]
Myo                            <-> Choban Elektrik                [International]
BenJamin Banger                <-> Opio                           [Hip-Hop]
Sro                            <-> Alex Mason                     [Instrumental]
Mutilation Rites               <-> Garage firm                    [International]
Alash Ensemble                 <-> Kokusai Bunka Shinkokai        [International]


## Deep pairs — where the slider is allowed to reach

The model ranks the whole catalogue by similarity. The manual evaluation only
ever looks at the top 5, so it says nothing about what the adventurous end of
the dial sounds like. These pairs come from rank 50 and rank 200 of the same
seeds' rankings.

The space is rebuilt here from scratch rather than imported, so this notebook
does not depend on either of our `modeling/` packages.

In [5]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

print("loading features (~900MB, takes a few seconds)")
features = pd.read_csv(META_DIR / "features.csv", index_col=0, header=[0, 1, 2])
X = features.loc[small.index].dropna()

X_scaled = StandardScaler().fit_transform(X)
Z = PCA(n_components=0.90, random_state=SEED).fit_transform(X_scaled)
print(f"{X.shape[0]} tracks, {X.shape[1]} features -> {Z.shape[1]} components")

DEPTHS = [50, 200]
index = NearestNeighbors(metric="cosine").fit(Z)
positions = {t: i for i, t in enumerate(X.index)}

deep_pairs = []
for s in SEEDS:
    _, idx = index.kneighbors(Z[[positions[s]]], n_neighbors=max(DEPTHS) + 2)
    ranked = [int(X.index[i]) for i in idx[0][1:]]  # drop the seed itself
    for depth in DEPTHS:
        deep_pairs.append({"seed": s, "other": ranked[depth - 1], "depth": depth})

print(f"\n{len(deep_pairs)} deep pairs at ranks {DEPTHS}")

loading features (~900MB, takes a few seconds)


8000 tracks, 518 features -> 146 components

20 deep pairs at ranks [50, 200]


## Assembling the pool

Every pair gets a blind id, and which clip plays first is decided by a coin
flip. Always putting the seed first would let a rater work out the structure
after a few pairs, and position on its own biases judgement.

In [6]:
rows = []
for s, recs in MODEL_PAIRS.items():
    for rank, other in enumerate(recs, start=1):
        rows.append({"kind": "model", "seed": s, "other": other, "rank": rank})
for s, other in random_pairs.items():
    rows.append({"kind": "random", "seed": s, "other": other, "rank": None})
for p in deep_pairs:
    rows.append({"kind": "deep", "seed": p["seed"], "other": p["other"], "rank": p["depth"]})

pool_df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
pool_df["pair_id"] = [f"P{i:02d}" for i in range(1, len(pool_df) + 1)]

flip = rng.random(len(pool_df)) < 0.5
pool_df["track_a"] = np.where(flip, pool_df["other"], pool_df["seed"])
pool_df["track_b"] = np.where(flip, pool_df["seed"], pool_df["other"])

print(pool_df["kind"].value_counts().to_string())
print(f"\n{len(pool_df)} pairs total")

kind
model     50
deep      20
random    10

80 pairs total


## Matching the loudness

People hear two clips at the same volume as more alike than two clips at
different volumes, whatever the music is doing. Left alone, that would come
back to us as similarity.

It is not a small effect here: the loudest clip in the pool carries 20x the RMS
energy of the quietest, because FMA is thousands of independent uploads with no
shared mastering.

No audio processing is needed to correct it — `features.csv` already carries
each track's RMS energy, so the form can simply play each clip at its own
volume. The reference is the 5th percentile rather than the true minimum: one
unusually quiet track would otherwise drag every other clip down with it. Only
about 2% of clips are loud enough to hit the floor.

Everything is attenuated, never amplified, so no clip is ever pushed into
distortion. The rater turns their own volume up once at the start.

In [7]:
rms = X[("rmse", "mean", "01")]
used = sorted(set(pool_df["track_a"]) | set(pool_df["track_b"]))

REFERENCE, FLOOR = 0.05, 0.15
target = rms.loc[used].quantile(REFERENCE)
volume = {int(t): float(np.clip(target / rms.loc[t], FLOOR, 1.0)) for t in used}

values = np.array(list(volume.values()))
print(f"{len(volume)} clips, loudest / quietest = {rms.loc[used].max() / rms.loc[used].min():.0f}x")
print(f"volume range  : {values.min():.2f} - {values.max():.2f}  (median {np.median(values):.2f})")
print(f"hitting floor : {(values <= FLOOR + 1e-9).mean():.0%}")

90 clips, loudest / quietest = 20x
volume range  : 0.15 - 1.00  (median 0.32)
hitting floor : 2%


## Copying the clips

Copied, not moved or converted — the originals are read-only and stay exactly
as they are. Names carry the pair id only.

A track that appears in several pairs is copied once per pair, under each pair's
own name. That costs some disk space and buys the blinding: one shared filename
would tell a rater that two pairs have a track in common.

In [8]:
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

total = 0
for row in pool_df.itertuples():
    for side, track_id in (("A", row.track_a), ("B", row.track_b)):
        destination = CLIPS_DIR / f"{row.pair_id}_{side}.mp3"
        shutil.copyfile(audio_path(int(track_id)), destination)
        total += destination.stat().st_size

print(f"{len(pool_df) * 2} clips written to {CLIPS_DIR}")
print(f"{total / 1e6:.0f} MB")

160 clips written to ../data/listening_test/clips
157 MB


## Who rates what

The pool is 80 pairs — around 80 minutes of listening, far more than a
volunteer will finish, and an abandoned form is worse than a short one. So the
pool is split rather than shortened. Each rater gets about 21 pairs, roughly 20
minutes:

- a **core** of ~10 pairs that *everyone* rates. Agreement between raters can
  only be measured on pairs more than one person heard, and that agreement is
  the ceiling on how well any model could match human judgement.
- **12 more from a rotation**, sized so that six raters cover every remaining
  pair exactly once. Pairs rated by one person are still worth having: the
  correlation between human scores and our computed metrics is built from the
  whole pool, and it gains more from breadth than from repetition.

The core holds model, random and deep pairs in proportion, so the one set
everybody hears is not accidentally all of one kind.

In [9]:
N_RATERS = 6
CORE_SIZE = 10

# proportional to each kind's share of the pool, so the set everybody hears is
# not accidentally all model pairs
core = []
for kind, group in pool_df.groupby("kind"):
    share = max(1, round(CORE_SIZE * len(group) / len(pool_df)))
    core += group.sample(share, random_state=SEED)["pair_id"].tolist()

rest = [str(p) for p in rng.permutation([p for p in pool_df["pair_id"] if p not in core])]

# big enough that six raters cover the whole remainder — a pair nobody hears is
# a pair we paid for in preparation and got nothing back from
ROTATION_SIZE = -(-len(rest) // N_RATERS)  # ceiling division

assignments = []
for r in range(N_RATERS):
    share = rest[r * ROTATION_SIZE:(r + 1) * ROTATION_SIZE]
    for pair_id in core + share:
        assignments.append({"rater": f"R{r + 1}", "pair_id": pair_id,
                            "in_core": pair_id in core})

assignments_df = pd.DataFrame(assignments)
per_rater = assignments_df.groupby("rater").size()
print(f"{N_RATERS} raters, {per_rater.min()}-{per_rater.max()} pairs each")
print(f"core rated by everyone : {len(core)}")
print(f"pool covered           : {assignments_df['pair_id'].nunique()} of {len(pool_df)}")

6 raters, 20-21 pairs each
core rated by everyone : 9
pool covered           : 80 of 80


## Writing it out

Two files with different audiences, and they must not be mixed up.

`manifest.csv` is what the rating form is built from: pair id, two filenames,
two volumes. Nothing in it reveals what a pair is.

`pairs_key.csv` is the answer key. It is only opened *after* the ratings come
back. Giving it to a rater, or leaving it in the folder that gets shared,
invalidates the whole test.

In [10]:
manifest = pd.DataFrame({
    "pair_id": pool_df["pair_id"],
    "clip_a": pool_df["pair_id"] + "_A.mp3",
    "clip_b": pool_df["pair_id"] + "_B.mp3",
    "volume_a": [round(volume[int(t)], 3) for t in pool_df["track_a"]],
    "volume_b": [round(volume[int(t)], 3) for t in pool_df["track_b"]],
})
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
assignments_df.to_csv(OUT_DIR / "assignments.csv", index=False)

key = pool_df.copy()
for side in ("seed", "other"):
    meta = pd.DataFrame([label(int(t)) for t in key[side]], index=key.index)
    key[[f"{side}_{c}" for c in meta.columns]] = meta
key["in_core"] = key["pair_id"].isin(core)
key.to_csv(OUT_DIR / "pairs_key.csv", index=False)

print("written:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name:18} {f.stat().st_size / 1024:6.1f} KB")
print(f"  clips/            {len(list(CLIPS_DIR.glob('*.mp3')))} files")

manifest.head()

written:
  assignments.csv       1.6 KB
  manifest.csv          2.8 KB
  pairs_key.csv         9.7 KB
  clips/            160 files


,pair_id,clip_a,clip_b,volume_a,volume_b
0,P01,P01_A.mp3,P01_B.mp3,0.332,0.382
1,P02,P02_A.mp3,P02_B.mp3,0.447,0.489
2,P03,P03_A.mp3,P03_B.mp3,0.189,0.220
3,P04,P04_A.mp3,P04_B.mp3,0.554,0.332
4,P05,P05_A.mp3,P05_B.mp3,0.181,0.269


## One bundle per rater

The whole pool is around 150 MB of audio, which nobody is going to download to
do a favour. A rater only needs their own pairs — about 40 MB.

This writes one folder per rater holding just their clips and their own
manifest, so each person gets one link and one short form. It duplicates audio
on disk, which costs nothing here and is what makes the test actually
distributable.

In [11]:
for rater, group in assignments_df.groupby("rater"):
    folder = OUT_DIR / "by_rater" / rater
    (folder / "clips").mkdir(parents=True, exist_ok=True)

    mine = manifest[manifest["pair_id"].isin(group["pair_id"])]
    mine.to_csv(folder / "manifest.csv", index=False)

    for row in mine.itertuples():
        for clip in (row.clip_a, row.clip_b):
            shutil.copyfile(CLIPS_DIR / clip, folder / "clips" / clip)

    size = sum(f.stat().st_size for f in (folder / "clips").glob("*.mp3"))
    print(f"{rater}: {len(mine):2} pairs, {size / 1e6:3.0f} MB -> {folder}")

R1: 21 pairs,  42 MB -> ../data/listening_test/by_rater/R1
R2: 21 pairs,  41 MB -> ../data/listening_test/by_rater/R2
R3: 21 pairs,  43 MB -> ../data/listening_test/by_rater/R3


R4: 21 pairs,  41 MB -> ../data/listening_test/by_rater/R4


R5: 21 pairs,  44 MB -> ../data/listening_test/by_rater/R5
R6: 20 pairs,  42 MB -> ../data/listening_test/by_rater/R6


## The rating page

A CSV and a folder of mp3s is not something you hand to someone doing you a
favour. This writes a `rate.html` into each rater's folder: they open it, and
every pair is one row — play A, play B, click a score, move on.

It is a plain file, opened from the folder. No install, no account, no
internet. The clips sit next to it, so the two travel together as one download.

Three things it does that matter for the result:

- **plays each clip at its own volume**, from the manifest, so loudness
  differences do not read as similarity
- **saves as you go** to the browser's own storage, so a half-finished session
  survives a closed tab. Nobody restarts a 20 minute task
- **gives back a CSV** at the end — downloaded, or copied out of a text box if
  the download is blocked

Nothing in the page names an artist, a title, a genre or a model score. The
rater sees `P34`, two play buttons, and five numbers.

In [12]:
RATING_PAGE = """<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Listening test __RATER__</title>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=IBM+Plex+Sans:wght@400;500;600&display=swap">
<style>
  :root {
    --ground: #f7f6f3; --surface: #ffffff; --ink: #1b1d1c; --muted: #6d7370;
    --line: #e0dfd9; --line-strong: #c9c8c0;
    --accent: #2d5b4e; --accent-ink: #ffffff; --accent-wash: #e6efeb;
    --sans: "IBM Plex Sans", ui-sans-serif, system-ui, -apple-system, sans-serif;
    --mono: "IBM Plex Mono", ui-monospace, "SF Mono", Menlo, monospace;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --ground: #14171a; --surface: #1c2023; --ink: #e7eae8; --muted: #8e9793;
      --line: #2b3034; --line-strong: #3d4449;
      --accent: #7fb7a1; --accent-ink: #10201a; --accent-wash: #1f2d28;
    }
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--ground); color: var(--ink);
         font-family: var(--sans); font-size: 15px; line-height: 1.55; }
  .page { max-width: 780px; margin: 0 auto; padding-inline: 20px; padding-block: 40px 96px; }
  h1 { font-size: 26px; font-weight: 600; margin: 0 0 6px; letter-spacing: -0.01em; }
  .standfirst { color: var(--muted); margin: 0 0 32px; max-width: 58ch; }
  .standfirst strong { color: var(--ink); font-weight: 500; }
  .brief { background: var(--surface); border: 1px solid var(--line); border-radius: 4px;
           padding: 20px 22px; margin-bottom: 14px; }
  .eyebrow { font-family: var(--mono); font-size: 11px; text-transform: uppercase;
             letter-spacing: 0.09em; color: var(--muted); margin: 0 0 14px; }
  .scale { margin: 0; display: grid; grid-template-columns: 22px 1fr; gap: 7px 14px; }
  .scale dt { font-family: var(--mono); font-weight: 500; color: var(--accent); }
  .scale dd { margin: 0; }
  .scale dd span { color: var(--muted); }
  .howto { list-style: none; margin: 0; padding: 0; display: grid; gap: 8px; }
  .howto li { display: grid; grid-template-columns: 22px 1fr; gap: 14px; color: var(--muted); }
  .howto li::before { content: attr(data-step); font-family: var(--mono); font-size: 11px;
                      color: var(--accent); padding-top: 3px; }
  .meter { position: sticky; top: 0; z-index: 10; background: var(--ground);
           padding-block: 14px 12px; margin-block: 26px 4px; border-bottom: 1px solid var(--line); }
  .meter-row { display: flex; align-items: baseline; justify-content: space-between; gap: 12px; }
  .meter-row b { font-family: var(--mono); font-weight: 500; font-variant-numeric: tabular-nums; }
  .meter-row span { color: var(--muted); font-size: 13px; }
  .rail { height: 3px; background: var(--line); margin-top: 9px; }
  .rail i { display: block; height: 100%; width: 0; background: var(--accent); transition: width .25s; }
  .pair { border-bottom: 1px solid var(--line); padding-block: 18px; display: grid; gap: 12px; }
  .pair-main { display: flex; flex-wrap: wrap; align-items: center; gap: 10px; }
  .pid { font-family: var(--mono); font-size: 13px; color: var(--muted); width: 40px; flex: none; }
  .pair.rated .pid { color: var(--accent); }
  .clip { flex: none; width: 120px; }
  .clip button { width: 100%; display: flex; align-items: center; gap: 8px; font: inherit;
                 font-size: 14px; color: var(--ink); background: var(--surface);
                 border: 1px solid var(--line-strong); border-radius: 3px;
                 padding: 8px 12px; cursor: pointer; }
  .clip button:hover { border-color: var(--accent); }
  .clip button.on { background: var(--accent); color: var(--accent-ink); border-color: var(--accent); }
  .clip button .glyph { font-size: 11px; }
  .clip button .side { font-family: var(--mono); font-weight: 500; }
  .clip button .heard { margin-left: auto; font-family: var(--mono); font-size: 11px;
                        font-variant-numeric: tabular-nums; opacity: .65; }
  .clip .played { height: 2px; background: var(--line); }
  .clip .played i { display: block; height: 100%; width: 0; background: var(--accent); }
  .scores { display: flex; gap: 4px; margin-left: auto; }
  .scores button { width: 38px; padding: 8px 0; font-family: var(--mono); font-size: 14px;
                   color: var(--muted); background: var(--surface);
                   border: 1px solid var(--line); border-radius: 3px; cursor: pointer; }
  .scores button:hover { border-color: var(--accent); color: var(--ink); }
  .scores button.on { background: var(--accent); color: var(--accent-ink); border-color: var(--accent); }
  .why { display: flex; flex-wrap: wrap; gap: 5px; padding-left: 50px; }
  .why button { font: inherit; font-size: 12.5px; color: var(--muted); background: transparent;
                border: 1px solid var(--line); border-radius: 100px; padding: 3px 11px; cursor: pointer; }
  .why button.on { background: var(--accent-wash); color: var(--accent); border-color: var(--accent); }
  button:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }
  .finish { margin-top: 36px; background: var(--surface); border: 1px solid var(--line);
            border-radius: 4px; padding: 22px; }
  .finish h2 { font-size: 17px; font-weight: 600; margin: 0 0 6px; }
  .finish p { color: var(--muted); margin: 0 0 16px; max-width: 54ch; }
  .actions { display: flex; flex-wrap: wrap; gap: 8px; }
  .actions button { font: inherit; font-size: 14px; border-radius: 3px; padding: 9px 16px;
                    cursor: pointer; border: 1px solid var(--line-strong);
                    background: var(--surface); color: var(--ink); }
  .actions button.primary { background: var(--accent); color: var(--accent-ink); border-color: var(--accent); }
  textarea { width: 100%; min-height: 150px; margin-top: 14px; font-family: var(--mono);
             font-size: 12px; background: var(--ground); color: var(--ink);
             border: 1px solid var(--line); border-radius: 3px; padding: 12px; resize: vertical; }
  .note { font-size: 13px; color: var(--muted); margin-top: 12px; }
  @media (max-width: 560px) { .scores { margin-left: 50px; } .clip { width: calc(50% - 5px); } }
  @media (prefers-reduced-motion: reduce) { .rail i, .clip .played i { transition: none; } }
</style>
</head>
<body>
<div class="page">
  <p class="eyebrow" style="margin-bottom:10px">Set __RATER__</p>
  <h1>Listening test</h1>
  <p class="standfirst">
    __N__ pairs of 30-second clips. For each one, say how <strong>alike they sound</strong> &mdash;
    not which you prefer. About twenty minutes.
  </p>

  <div class="brief">
    <p class="eyebrow">The scale</p>
    <dl class="scale">
      <dt>1</dt><dd><b>Unrelated.</b> <span>No meaningful musical similarity.</span></dd>
      <dt>2</dt><dd><b>Weak.</b> <span>One or two minor characteristics connect them.</span></dd>
      <dt>3</dt><dd><b>Moderate.</b> <span>A recognisable connection, with substantial differences.</span></dd>
      <dt>4</dt><dd><b>Strong.</b> <span>Several important characteristics clearly connect them.</span></dd>
      <dt>5</dt><dd><b>Very strong.</b> <span>Highly alike in sound, atmosphere or production.</span></dd>
    </dl>
  </div>

  <div class="brief">
    <p class="eyebrow">Before you start</p>
    <ul class="howto">
      <li data-step="01">Use headphones if you have them, set your volume on the first pair, then leave it alone. The clips are already levelled.</li>
      <li data-step="02">Listen to roughly the same amount of each side &mdash; the counter on each button shows how long you have heard.</li>
      <li data-step="03">Unsure? Play them again. A considered 3 is worth more than a quick guess.</li>
    </ul>
  </div>

  <div class="meter">
    <div class="meter-row">
      <b><span id="done">0</span> / __N__ rated</b>
      <span id="left">__N__ to go</span>
    </div>
    <div class="rail"><i id="fill"></i></div>
  </div>

  <div id="pairs"></div>

  <div class="finish">
    <h2>Sending your answers back</h2>
    <p>Download the file and send it back, or copy the text if the download does not work.
       It holds only pair numbers and scores.</p>
    <div class="actions">
      <button class="primary" id="save">Download __RATER___ratings.csv</button>
      <button id="copy">Copy instead</button>
    </div>
    <textarea id="out" hidden readonly aria-label="Your answers as text"></textarea>
    <p class="note" id="status"></p>
  </div>
</div>

<script>
const RATER = "__RATER__";
const PAIRS = __DATA__;
const REASONS = ["rhythm","timbre","mood","instrumentation","energy","production",
                 "harmony","vocals","nothing"];
const STORE = "listening-" + RATER;

let answers = {};
try { answers = JSON.parse(localStorage.getItem(STORE) || "{}"); } catch (e) { answers = {}; }

const audio = new Audio();
let current = null;

function clock(s) {
  s = Math.floor(s || 0);
  return Math.floor(s / 60) + ":" + String(s % 60).padStart(2, "0");
}

function save() {
  try { localStorage.setItem(STORE, JSON.stringify(answers)); } catch (e) {}
  const n = PAIRS.filter(p => answers[p.pair_id] && answers[p.pair_id].score).length;
  document.getElementById("done").textContent = n;
  document.getElementById("left").textContent =
    n === PAIRS.length ? "all done \\u2014 send your answers below" : (PAIRS.length - n) + " to go";
  document.getElementById("fill").style.width = (100 * n / PAIRS.length) + "%";
}

audio.addEventListener("timeupdate", () => {
  if (!current) return;
  current.bar.style.width = (100 * audio.currentTime / (audio.duration || 30)) + "%";
  current.heard.textContent = clock(audio.currentTime);
});
audio.addEventListener("ended", () => {
  if (current) current.button.classList.remove("on");
  current = null;
});

function toggle(clip, file, vol) {
  if (current && current.button === clip.button) {
    audio.pause(); clip.button.classList.remove("on"); current = null; return;
  }
  if (current) current.button.classList.remove("on");
  audio.src = "clips/" + file;
  audio.volume = vol;
  audio.play().catch(() => {});
  clip.button.classList.add("on");
  current = clip;
}

function clipControl(side, file, vol) {
  const wrap = document.createElement("div");
  wrap.className = "clip";
  const button = document.createElement("button");
  button.type = "button";
  button.innerHTML = '<span class="glyph">&#9654;</span><span class="side">' + side +
                     '</span><span class="heard">0:00</span>';
  const track = document.createElement("div");
  track.className = "played";
  const bar = document.createElement("i");
  track.appendChild(bar);
  const clip = { button, bar, heard: button.querySelector(".heard") };
  button.addEventListener("click", () => toggle(clip, file, vol));
  wrap.append(button, track);
  return wrap;
}

const host = document.getElementById("pairs");

for (const pair of PAIRS) {
  const saved = answers[pair.pair_id] || (answers[pair.pair_id] = { score: null, why: [] });
  const row = document.createElement("section");
  row.className = "pair" + (saved.score ? " rated" : "");

  const main = document.createElement("div");
  main.className = "pair-main";
  const id = document.createElement("span");
  id.className = "pid";
  id.textContent = pair.pair_id;
  main.appendChild(id);
  main.appendChild(clipControl("A", pair.clip_a, pair.volume_a));
  main.appendChild(clipControl("B", pair.clip_b, pair.volume_b));

  const scores = document.createElement("div");
  scores.className = "scores";
  scores.setAttribute("role", "group");
  scores.setAttribute("aria-label", "Similarity for " + pair.pair_id);
  for (let n = 1; n <= 5; n++) {
    const b = document.createElement("button");
    b.type = "button";
    b.textContent = n;
    if (saved.score === n) b.classList.add("on");
    b.addEventListener("click", () => {
      saved.score = n;
      [...scores.children].forEach(c => c.classList.toggle("on", c === b));
      row.classList.add("rated");
      save();
    });
    scores.appendChild(b);
  }
  main.appendChild(scores);
  row.appendChild(main);

  const why = document.createElement("div");
  why.className = "why";
  for (const reason of REASONS) {
    const b = document.createElement("button");
    b.type = "button";
    b.textContent = reason;
    if (saved.why.includes(reason)) b.classList.add("on");
    b.addEventListener("click", () => {
      const at = saved.why.indexOf(reason);
      if (at < 0) saved.why.push(reason); else saved.why.splice(at, 1);
      b.classList.toggle("on");
      save();
    });
    why.appendChild(b);
  }
  row.appendChild(why);
  host.appendChild(row);
}

save();

function asText() {
  const lines = ["rater,pair_id,similarity_1_5,connects"];
  for (const p of PAIRS) {
    const a = answers[p.pair_id] || {};
    lines.push([RATER, p.pair_id, a.score || "", (a.why || []).join(" ")].join(","));
  }
  return lines.join("\\n");
}

const status = document.getElementById("status");

document.getElementById("save").addEventListener("click", () => {
  const blob = new Blob([asText()], { type: "text/csv" });
  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob);
  a.download = RATER + "_ratings.csv";
  a.click();
  status.textContent = "Saved to your downloads folder. Send that file back.";
});

document.getElementById("copy").addEventListener("click", async () => {
  const text = asText();
  try {
    await navigator.clipboard.writeText(text);
    status.textContent = "Copied. Paste it into your reply.";
  } catch (e) {
    const box = document.getElementById("out");
    box.value = text; box.hidden = false; box.select();
    status.textContent = "Select the text above and copy it.";
  }
});
</script>
</body>
</html>
"""

WORDS = {20: "Twenty", 21: "Twenty-one"}

for rater, group in assignments_df.groupby("rater"):
    folder = OUT_DIR / "by_rater" / rater
    mine = manifest[manifest["pair_id"].isin(group["pair_id"])]
    page = (RATING_PAGE
            .replace("__DATA__", json.dumps(mine.to_dict("records")))
            .replace("__RATER__", rater)
            .replace("__N__ pairs of 30-second", WORDS[len(mine)] + " pairs of 30-second")
            .replace("__N__", str(len(mine))))
    (folder / "rate.html").write_text(page, encoding="utf-8")

print("rate.html written into each rater folder")
print(f"\nto check it yourself:\n  open {OUT_DIR / 'by_rater' / 'R2' / 'rate.html'}")

rate.html written into each rater folder

to check it yourself:
  open ../data/listening_test/by_rater/R2/rate.html


## What the rater is asked

Keeping the rubric from the baseline's manual evaluation, so the scores we
already have and the ones we collect now mean the same thing.

**How similar do these two tracks sound? (1-5)**

| | |
|---|---|
| 1 | No meaningful musical similarity — they feel unrelated |
| 2 | Weak. One or two minor characteristics connect them |
| 3 | Moderate. A recognisable connection, with substantial differences |
| 4 | Strong. Several important characteristics clearly connect them |
| 5 | Very strong. Highly alike in sound, atmosphere or production |

**What connects them?** (optional, pick any)
rhythm · timbre · mood · instrumentation · energy · production · harmony ·
vocals · nothing

Do **not** ask which track they preferred. Liking is taste and varies by
person; hearing two tracks as alike is far more consistent between people, and
it is the only one of the two this recommender is trying to get right.

### Next

1. Build one form from `manifest.csv` — one form for everyone, or the ratings
   cannot be pooled.
2. Share `clips/` and the form. Never share `pairs_key.csv`.
3. When the ratings come back, join them to the key and read, in this order:
   model pairs against random pairs; agreement between raters; then which
   computed metric — cosine, brightness difference, tempo difference, genre
   match — actually tracks the human scores.